New ML test pipeline script


Import all libraries needed for this ML script.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
print('Imports Successfully!')

Imports Successfully!


In [3]:
#to check the python version and the path of the python executable
import sys
print(sys.executable)

/Users/boracomert/Desktop/Osint_project/.venv/bin/python


the next step: Load data 

In [4]:
DATA_PATH = "../data/processed/labeled_merged_data_cleaned.csv"

df = pd.read_csv(DATA_PATH)
print('Data Loaded Successfully!')

#the model shouldnt see layoff information becuse it can cause data leakage, so we will drop the layoff columns from the features
META_COLS = ['company', 'date', 'quarter', 'layoff','same_quarter', 'next_quarter', 'layoff_same_quarter', 'layoff_next_quarter', 'layoff_same_or_next_quarter']

FEATURE_COLS = [col for col in df.columns if col not in META_COLS]


n_companies  = df['company'].nunique()
n_rows_pos   = len(df)

print(f'Positive companies : {n_companies}')
print(f'Positive rows      : {n_rows_pos}')
print(f'Feature columns    : {len(FEATURE_COLS)}')
print(f'Quarters present   : {sorted(df["quarter"].unique())}')
df.head(5)

Data Loaded Successfully!
Positive companies : 1941
Positive rows      : 12047
Feature columns    : 346
Quarters present   : ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']


,company,date,quarter,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,...,fin_Other Non Interest Expense,fin_Depletion Income Statement,fin_Policyholder Benefits Ceded,fin_Net Income Extraordinary,same_quarter,next_quarter,layoff_same_quarter,layoff_next_quarter,layoff_same_or_next_quarter,layoff
0,AFCONS.BO,2024-09-30,2024Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q3,2024Q4,0,0,0,0
1,AFCONS.BO,2024-12-31,2024Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q4,2025Q1,0,0,0,0
2,AFCONS.BO,2025-03-31,2025Q1,367784631.0,367784631.0,1.795550e+10,2.343300e+10,5.259830e+10,7.496240e+10,3.021220e+10,...,NaN,NaN,NaN,NaN,2025Q1,2025Q2,0,0,0,0
3,AFCONS.BO,2025-06-30,2025Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2025Q2,2025Q3,0,0,0,0
4,AFCONS.BO,2025-09-30,2025Q3,367784631.0,367784631.0,3.097260e+10,3.565230e+10,5.388340e+10,8.861110e+10,3.037610e+10,...,NaN,NaN,NaN,NaN,2025Q3,2025Q4,0,0,0,0


In [9]:
# there are many feature cols with almost all nAn values, we will drop those cols

threshold = 0.9
cols_to_drop = [col for col in FEATURE_COLS if df[col].isna().mean() > threshold]
print(f'Columns to drop due to high NaN percentage (> {threshold*100}%): {cols_to_drop}')
FEATURE_COLS = [col for col in FEATURE_COLS if col not in cols_to_drop]
print(f'Updated feature columns count: {len(FEATURE_COLS)}')
print(df[FEATURE_COLS].isna().sum().sum(), 'NaN values remaining in feature columns after dropping high NaN columns.')


# now fill remanining Nan values with mean of the column with this code 
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].mean())

#now check for zero variance columns, as they do not provide any useful information for the model and can be dropped
variance = df[FEATURE_COLS].var()
zero_variance_cols = variance[variance == 0].index.tolist()
print(f'Columns with zero variance: {len(zero_variance_cols)}')


Columns to drop due to high NaN percentage (> 90.0%): []
Updated feature columns count: 227
1528047 NaN values remaining in feature columns after dropping high NaN columns.
Columns with zero variance: 0


In [10]:

# Now we will check the correlation between the feature columns, feature columns with high multicolinearity can cause issues for some models,such as KNN 
corr_matrix = df[FEATURE_COLS].corr()
corr_matrix


,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,bs_Net Tangible Assets,bs_Capital Lease Obligations,bs_Common Stock Equity,...,fin_Interest Income Non Operating,fin_Research And Development,fin_Write Off,fin_Restructuring And Mergern Acquisition,fin_Depreciation Amortization Depletion Income Statement,fin_Amortization,fin_Other Special Charges,fin_Impairment Of Capital Assets,fin_Selling And Marketing Expense,fin_Total Other Finance Cost
bs_Ordinary Shares Number,1.000000,0.998371,0.067281,0.174818,0.088343,0.106286,0.119967,0.088319,0.049725,0.088985,...,0.074047,0.053813,0.364471,0.000694,0.090407,0.007096,0.292158,0.467743,0.098639,-0.003933
bs_Share Issued,0.998371,1.000000,0.060771,0.168324,0.088440,0.105379,0.121396,0.088415,0.047987,0.088870,...,0.074811,0.053377,0.360371,0.038318,0.081099,0.006341,0.288495,0.468000,0.098382,-0.004195
bs_Net Debt,0.067281,0.060771,1.000000,0.922397,0.017973,0.167499,-0.104213,0.017966,0.156572,0.055529,...,0.007063,0.000670,0.011717,0.000410,0.192780,0.004883,0.454977,0.001117,0.005041,0.024613
bs_Total Debt,0.174818,0.168324,0.922397,1.000000,0.372464,0.507864,0.261558,0.372456,0.430100,0.406266,...,0.264781,0.356611,0.045160,0.000247,0.220683,0.353855,0.579981,0.065071,0.354915,-0.250352
bs_Tangible Book Value,0.088343,0.088440,0.017973,0.372464,1.000000,0.988474,0.986488,0.999999,0.771148,0.999192,...,0.798510,0.995388,0.028519,0.026762,0.012914,0.990836,0.403031,0.036511,0.994508,-0.861303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fin_Amortization,0.007096,0.006341,0.004883,0.353855,0.990836,0.977273,0.971093,0.990840,0.759778,0.989830,...,0.797317,0.994231,0.002704,0.000434,0.021897,1.000000,0.374201,0.000088,0.987768,-0.846311
fin_Other Special Charges,0.292158,0.288495,0.454977,0.579981,0.403031,0.469437,0.346976,0.403019,0.419665,0.420905,...,0.266373,0.396677,0.498337,0.005132,-0.016616,0.374201,1.000000,0.041177,0.388524,-0.169269
fin_Impairment Of Capital Assets,0.467743,0.468000,0.001117,0.065071,0.036511,0.042233,0.057368,0.036499,0.014506,0.035867,...,0.021353,0.027031,0.049004,0.000158,0.005462,0.000088,0.041177,1.000000,0.053934,-0.015849
fin_Selling And Marketing Expense,0.098639,0.098382,0.005041,0.354915,0.994508,0.980960,0.976358,0.994509,0.769053,0.993598,...,0.795747,0.991922,0.031126,0.003904,0.001239,0.987768,0.388524,0.053934,1.000000,-0.870752


In [ ]:
#Now to train the models...

#target column is  layoff 

target_col = 'layoff'

x = df[FEATURE_COLS]
y = df[target_col]


X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    ('Logistic Regression', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'), True),
    ('Random Forest', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'), False),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=RANDOM_STATE,), False),
    ('XGBoost', XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'), False),
    ('SVM', SVC(random_state=RANDOM_STATE, probability=True), True),
    ('KNN', KNeighborsClassifier(), True),
    ('decision_tree', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'), False)
}

results = []




for model_name, model, needs_scaling in models:
    print(f'Training {model_name}...')
    
    if needs_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_proba)

    cv_auc = cross_val_score(model, X_train_scaled if needs_scaling else X_train, y_train, cv=cv, scoring='roc_auc').mean()

    
    results.append({
        'model': model_name,
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1-score': report['1']['f1-score'],
        'roc_auc': roc_auc,
        'cv_roc_auc': cv_auc
    })
    
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='roc_auc', ascending=False)
print(results_df.to_string(index=False))

SyntaxError: invalid syntax (3165289777.py, line 57)

In [ ]:
#visualisation etc